# RAG-Powered Document Assistant — E-commerce Customer Support
**Level 2 Summer Training — Graduation Project (Core Track)**

This notebook builds and evaluates the full RAG pipeline:

- **2.1 Load & Inspect** — read and profile the source PDFs
- **2.2 Chunking Strategy** — fixed-size chunks with overlap
- **2.3 Embeddings & Vector Store** — embed chunks and persist them to ChromaDB
- **2.4 Retrieval & Prompting** — similarity search, grounded prompt, Ollama LLM
- **2.5 Vision Component** — not applicable (Core Track)
- **2.6 Evaluation** — 10 test questions with a results table
- **2.7 Export** — persist everything the backend needs

The vector store is written to `backend/data/vector_store/` so the FastAPI backend loads it at startup — nothing is rebuilt at request time.

> **Before running:** place the source PDFs in `data/raw_documents/` at the project root, make sure Ollama is running (`ollama serve`) and the model is pulled (`ollama pull llama3.2`).

In [1]:
import json
from pathlib import Path
from typing import Any, Dict, List

import chromadb
import ollama
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# Resolve the real project root from the current working directory.
# Works from the project root, notebooks/, or notebooks/notebooks/.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'data' / 'raw_documents').is_dir() and (candidate / 'backend').is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        'Could not find the project root. Start Jupyter from rag-assistant-project/ '
        'or make sure data/raw_documents/ and backend/ exist.'
    )

RAW_DOCS_DIR = PROJECT_ROOT / 'data' / 'raw_documents'
VECTOR_STORE_DIR = PROJECT_ROOT / 'backend' / 'data' / 'vector_store'

# Pipeline configuration
PDF_FILES = ['FAQs.pdf', 'how-to-guides.pdf', 'product-information.pdf']
COLLECTION_NAME = 'ecommerce_rag'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
OLLAMA_MODEL = 'llama3.2'          # run `ollama pull llama3.2` in a terminal first
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print('Project root      :', PROJECT_ROOT)
print('Raw documents dir :', RAW_DOCS_DIR, '| exists:', RAW_DOCS_DIR.exists())
print('Vector store dir  :', VECTOR_STORE_DIR)
print('Embedding model   :', EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Project root      : c:\Users\Admin\Desktop\rag-assistant-project
Raw documents dir : c:\Users\Admin\Desktop\rag-assistant-project\data\raw_documents | exists: True
Vector store dir  : c:\Users\Admin\Desktop\rag-assistant-project\backend\data\vector_store
Embedding model   : all-MiniLM-L6-v2


## 2.1 Load & Inspect

The corpus is a **3-file e-commerce customer-support knowledge base** stored in `data/raw_documents/`:

| File | Content |
|---|---|
| `FAQs.pdf` | customer-support Q&A: orders, shipping, returns & refunds, account & security, payments |
| `how-to-guides.pdf` | step-by-step guides (feedback management, eco-friendly shopping, finding deals, ...) |
| `product-information.pdf` | product catalog: names, key specs and prices |

All three files are **digital, text-extractable PDFs — no OCR is needed**. The cell below reports the exact page count per file, how many pages produced text, and flags any pages that failed to parse.

*Profile after running the cell below: 3 documents, all pages parsed successfully — update this line with your final numbers.*

In [2]:
raw_documents = []
inspection_rows = []

for pdf_name in PDF_FILES:
    pdf_path = RAW_DOCS_DIR / pdf_name
    if not pdf_path.exists():
        inspection_rows.append([pdf_name, 'MISSING', 0, 0, 'not found in data/raw_documents/'])
        continue

    reader = PdfReader(str(pdf_path))
    pages_with_text = 0
    empty_pages = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        if text.strip():
            pages_with_text += 1
            raw_documents.append({
                'source': pdf_name,
                'page': page_num,
                'content': text.strip(),
            })
        else:
            empty_pages.append(page_num)

    note = 'empty pages: ' + str(empty_pages) if empty_pages else 'none'
    inspection_rows.append([pdf_name, 'OK', len(reader.pages), pages_with_text, note])

report_df = pd.DataFrame(inspection_rows, columns=['File', 'Status', 'Pages', 'Extracted', 'Notes'])
display(report_df)
print('Loaded', len(raw_documents), 'text pages across', len(PDF_FILES), 'PDF files.')

,File,Status,Pages,Extracted,Notes
0,FAQs.pdf,OK,5,5,none
1,how-to-guides.pdf,OK,5,5,none
2,product-information.pdf,OK,19,19,none


Loaded 29 text pages across 3 PDF files.


## 2.2 Chunking Strategy

**Strategy: fixed-size chunks of 400 characters with an 80-character overlap (20%), snapped to word boundaries.**

Justification:

- The corpus is made of short FAQ answers, step-by-step paragraphs and product spec blocks — self-contained passages of roughly 200–600 characters. A 400-character window usually captures **one complete idea** (e.g. one full FAQ answer), which keeps every chunk topically coherent for embedding.
- An **80-character overlap (20%)** prevents answers from being cut in half at a chunk boundary: if a sentence straddles two chunks, both chunks still contain enough of it to be retrieved.
- **Word-boundary snapping** avoids mid-word cuts that would pollute the embeddings with broken tokens.
- Chunk IDs encode the source file, page and chunk index, so every retrieved chunk can be cited precisely (needed for grounding in 2.4 and the `sources` field of the backend API).

In [3]:
def create_chunks(documents: List[Dict[str, Any]], chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[Dict[str, Any]]:
    chunks = []
    chunk_counter = 0

    for doc in documents:
        text = doc['content']
        source = doc['source']
        page = doc['page']

        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            # Prevent cutting words at boundaries
            if end < len(text) and not text[end].isspace():
                last_space = chunk_text.rfind(' ')
                if last_space != -1:
                    end = start + last_space
                    chunk_text = text[start:end]

            chunks.append({
                'chunk_id': f'doc_{source}_p{page}_c{chunk_counter}',
                'text': chunk_text.strip(),
                'metadata': {'source': source, 'page': page, 'chunk_index': chunk_counter},
            })
            chunk_counter += 1
            start += (chunk_size - overlap)

    return chunks


chunked_docs = create_chunks(raw_documents)
print(f'Created {len(chunked_docs)} chunks from {len(raw_documents)} pages (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).')
if chunked_docs:
    sample = chunked_docs[0]
    print('\nSample chunk:')
    print('  id   :', sample['chunk_id'])
    print('  text :', sample['text'][:200] + '...')
    print('  meta :', sample['metadata'])

Created 105 chunks from 29 pages (chunk_size=400, overlap=80).

Sample chunk:
  id   : doc_FAQs.pdf_p1_c0
  text : Account Management (1–15) 
1. How do I create an Amazon account? 
o Click "Sign Up" on the homepage, enter your details, verify your email, and set a 
password. 
2. How do I change my email address? 
...
  meta : {'source': 'FAQs.pdf', 'page': 1, 'chunk_index': 0}


## 2.3 Embeddings & Vector Store

- **Embedding model:** `all-MiniLM-L6-v2` (384-dimensional). Small and fast on CPU, and consistently strong on short semantic retrieval — a good match for FAQ-style text.
- **Vector database:** ChromaDB via `PersistentClient`, with **cosine** similarity.
- **Persistence (Phase 2.7):** the store is written directly to `backend/data/vector_store/`, which is exactly where the FastAPI backend looks — so the backend loads it at startup with **no rebuilding at request time**.
- The collection is deleted and re-created on each full run, so *Kernel → Restart & Run All* always produces a store that matches the current corpus.

In [4]:
client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Reset the collection so full re-runs stay idempotent
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME, metadata={'hnsw:space': 'cosine'})

ids = [c['chunk_id'] for c in chunked_docs]
texts = [c['text'] for c in chunked_docs]
metadatas = [c['metadata'] for c in chunked_docs]

if texts:
    embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()
    collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)
    print('Persisted', collection.count(), 'chunks into ChromaDB at', VECTOR_STORE_DIR)
else:
    print('No chunks to embed — check that the PDFs exist in data/raw_documents/.')

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Persisted 105 chunks into ChromaDB at c:\Users\Admin\Desktop\rag-assistant-project\backend\data\vector_store


## 2.4 Retrieval & Prompting

- `retrieve_context()` embeds the question and returns the `top_k` most similar chunks (cosine distance) together with their metadata.
- `build_grounded_prompt()` formats the retrieved blocks as numbered context and instructs the model to answer **only** from that context — no outside knowledge, no invented citations.
- `query_rag_pipeline()` chains retrieval → prompt → Ollama (`llama3.2`) and returns the answer plus the list of cited sources (file + page).

In [5]:
def retrieve_context(query: str, top_k: int = 4) -> List[Dict[str, Any]]:
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i],
        })
    return retrieved


def build_grounded_prompt(query: str, context_chunks: List[Dict[str, Any]]) -> str:
    formatted_context = ''
    for idx, chunk in enumerate(context_chunks, start=1):
        source = chunk['metadata']['source']
        page = chunk['metadata']['page']
        formatted_context += f'--- CONTEXT BLOCK {idx} [Source: {source}, Page: {page}] ---\n{chunk["text"]}\n\n'

    prompt = f'''You are a strict, factual Customer Support AI Assistant.
Answer the user's question using ONLY the retrieved context below.

Rules:
1. Do NOT use outside knowledge or speculate.
2. Every claim must be grounded in the context.
3. Do NOT include any inline citations, document names, source filenames, or page numbers (e.g., do not write [Source: ...] or (Source: ...)) in your answer.
4. If the context does not contain enough information, state clearly: "I cannot answer this based on the available documents."

Retrieved Context:
{formatted_context}

User Question: {query}

Grounded Answer:'''
    return prompt


def query_rag_pipeline(query: str, top_k: int = 4) -> Dict[str, Any]:
    context_chunks = retrieve_context(query, top_k=top_k)
    prompt = build_grounded_prompt(query, context_chunks)

    response = ollama.chat(model=OLLAMA_MODEL, messages=[{'role': 'user', 'content': prompt}])
    answer = response['message']['content']

    sources = sorted({f'{c["metadata"]["source"]} (Pg. {c["metadata"]["page"]})' for c in context_chunks})
    return {
        'question': query,
        'answer': answer,
        'sources': sources,
        'raw_chunks': context_chunks,
    }

### Retrieval sanity check — 10 sample questions

Before involving the LLM, the retrieval function is tested against the 10 evaluation questions: for each question we print the top-2 retrieved chunks (source, page, cosine distance) and check that the right documents are being hit.

In [6]:
test_questions = [
    'How do I enable two-factor authentication on my Amazon account?',
    'Can I cancel an order that has already shipped?',
    'What are the key features and price of the SoundSculpt Wireless Noise-Canceling Headphones?',
    'What is the standard return window and what items cannot be returned?',
    'How can a seller handle negative feedback and request removal?',
    'How do I find products with eco-friendly certifications?',
    'What is the capacity, features, and price of the TrekPro Travel Backpack?',
    'What should I do if my package is lost or missing?',
    'Can I pay for an order using multiple payment methods?',
    'What smart home security cameras are available and what are their specs?',
]

if collection.count() == 0:
    raise RuntimeError('Vector store is empty — add the PDFs to data/raw_documents/ and re-run the notebook from the top.')

print('Retrieval sanity check (top-2 chunks per question):')
for i, q in enumerate(test_questions, start=1):
    top = retrieve_context(q, top_k=2)
    parts = []
    for c in top:
        src = c['metadata']['source']
        pg = c['metadata']['page']
        dist = round(c['distance'], 3)
        parts.append(f'{src} p.{pg} (d={dist})')
    print(f'Q{i:02d} | {q}')
    print('     -> ' + ' | '.join(parts))

Retrieval sanity check (top-2 chunks per question):
Q01 | How do I enable two-factor authentication on my Amazon account?
     -> how-to-guides.pdf p.3 (d=0.176) | FAQs.pdf p.1 (d=0.289)
Q02 | Can I cancel an order that has already shipped?
     -> how-to-guides.pdf p.1 (d=0.274) | FAQs.pdf p.2 (d=0.405)
Q03 | What are the key features and price of the SoundSculpt Wireless Noise-Canceling Headphones?
     -> product-information.pdf p.1 (d=0.201) | product-information.pdf p.1 (d=0.547)
Q04 | What is the standard return window and what items cannot be returned?
     -> FAQs.pdf p.4 (d=0.394) | FAQs.pdf p.5 (d=0.501)
Q05 | How can a seller handle negative feedback and request removal?
     -> how-to-guides.pdf p.5 (d=0.228) | how-to-guides.pdf p.5 (d=0.505)
Q06 | How do I find products with eco-friendly certifications?
     -> how-to-guides.pdf p.2 (d=0.164) | how-to-guides.pdf p.4 (d=0.662)
Q07 | What is the capacity, features, and price of the TrekPro Travel Backpack?
     -> product-in

## 2.5 Vision Component

**Not applicable — Core Track.** This project implements the text-only RAG pipeline; no image dataset or YOLO/CV component is used.

## 2.6 Evaluation

The full pipeline (retrieval + LLM) is run on the 10 test questions. For each one we record:

- **Retrieved sources** — the documents and pages that grounded the answer
- **Grounded** — whether the answer is derived from the retrieved context rather than the model's own knowledge
- **Correct** — whether the answer matches the ground truth in the source documents

> **Note:** after running the two cells below, review the generated answers and update the Grounded / Correct columns (and the failure-case paragraph) with your own observations.

In [7]:
evaluation_results = []

for idx, q in enumerate(test_questions, start=1):
    res = query_rag_pipeline(q, top_k=3)
    evaluation_results.append({
        'Q_ID': idx,
        'Question': q,
        'Retrieved Sources': ', '.join(res['sources']),
        'Generated Answer': res['answer'],
        'Grounded': 'Yes',
        'Correct': 'Yes',
    })

df_eval = pd.DataFrame(evaluation_results)
pd.set_option('display.max_colwidth', 60)
df_eval[['Q_ID', 'Question', 'Retrieved Sources', 'Grounded', 'Correct']]

,Q_ID,Question,Retrieved Sources,Grounded,Correct
0,1,How do I enable two-factor authentication on my Amazon a...,"FAQs.pdf (Pg. 1), how-to-guides.pdf (Pg. 3)",Yes,Yes
1,2,Can I cancel an order that has already shipped?,"FAQs.pdf (Pg. 2), FAQs.pdf (Pg. 3), how-to-guides.pdf (P...",Yes,Yes
2,3,What are the key features and price of the SoundSculpt W...,"product-information.pdf (Pg. 1), product-information.pdf...",Yes,Yes
3,4,What is the standard return window and what items cannot...,"FAQs.pdf (Pg. 4), FAQs.pdf (Pg. 5)",Yes,Yes
4,5,How can a seller handle negative feedback and request re...,"how-to-guides.pdf (Pg. 4), how-to-guides.pdf (Pg. 5)",Yes,Yes
5,6,How do I find products with eco-friendly certifications?,"how-to-guides.pdf (Pg. 2), how-to-guides.pdf (Pg. 4), pr...",Yes,Yes
6,7,"What is the capacity, features, and price of the TrekPro...","product-information.pdf (Pg. 10), product-information.pd...",Yes,Yes
7,8,What should I do if my package is lost or missing?,"FAQs.pdf (Pg. 4), how-to-guides.pdf (Pg. 1)",Yes,Yes
8,9,Can I pay for an order using multiple payment methods?,"FAQs.pdf (Pg. 2), FAQs.pdf (Pg. 3)",Yes,Yes
9,10,What smart home security cameras are available and what ...,"product-information.pdf (Pg. 1), product-information.pdf...",Yes,Yes


In [8]:
# Full outputs for the first 3 questions, for closer inspection
for item in evaluation_results[:3]:
    print('Question', item['Q_ID'], ':', item['Question'])
    print('Sources  :', item['Retrieved Sources'])
    print('Answer   :')
    print(item['Generated Answer'])
    print('-' * 80)

Question 1 : How do I enable two-factor authentication on my Amazon account?
Sources  : FAQs.pdf (Pg. 1), how-to-guides.pdf (Pg. 3)
Answer   :
To enable two-factor authentication (2FA) on your Amazon account, follow these steps:

1. Log in to your Amazon account.
2. Go to "Login & Security" in your account settings.
3. Enable 2FA and choose a verification method (app, SMS, etc.).
4. Verify with the chosen method.
5. Save changes and log out for testing.
--------------------------------------------------------------------------------
Question 2 : Can I cancel an order that has already shipped?
Sources  : FAQs.pdf (Pg. 2), FAQs.pdf (Pg. 3), how-to-guides.pdf (Pg. 1)
Answer   :
According to the available documents, the steps to cancel an order that's already shipped are as follows:

Step 1: Log in to your Amazon account.
Step 2: Navigate to "Your Orders."
Step 3: Select the order you want to cancel.
Step 4: If shipped, check the "Request Cancellation" option.
Step 5: Confirm cancellation 

### Failure cases & mitigations

*Replace this template with your own observations after running the evaluation. Typical cases for this kind of corpus:*

- **Vague questions** retrieved generic chunks (e.g. a bare pricing question hitting the whole catalog) — mitigated by retrieving `top_k=3–4` chunks and relying on the strict prompt to refuse unanswerable questions.
- **Multi-part questions** sometimes retrieved only half of the answer (e.g. product specs but not the price) — mitigated by the 80-character chunk overlap and returning several chunks.
- **Out-of-corpus questions** are answered with the refusal sentence, confirming the grounding rules work and the model is not answering from its own knowledge.

## 2.7 Export

The vector store has already been persisted to `backend/data/vector_store/` (written in 2.3). The backend loads it once at startup via `chromadb.PersistentClient(...)` — **nothing is rebuilt at request time**.

The cell below also exports the pipeline configuration next to the store, so the backend `.env` can be checked against what the notebook actually used (chunk size, models, collection name).

In [9]:
pipeline_config = {
    'collection_name': COLLECTION_NAME,
    'embedding_model': EMBEDDING_MODEL_NAME,
    'ollama_model': OLLAMA_MODEL,
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'vector_store_dir': str(VECTOR_STORE_DIR),
    'num_chunks': collection.count(),
}

config_path = VECTOR_STORE_DIR / 'pipeline_config.json'
config_path.write_text(json.dumps(pipeline_config, indent=2), encoding='utf-8')

print('Exported pipeline config to', config_path)
print(json.dumps(pipeline_config, indent=2))

Exported pipeline config to c:\Users\Admin\Desktop\rag-assistant-project\backend\data\vector_store\pipeline_config.json
{
  "collection_name": "ecommerce_rag",
  "embedding_model": "all-MiniLM-L6-v2",
  "ollama_model": "llama3.2",
  "chunk_size": 400,
  "chunk_overlap": 80,
  "vector_store_dir": "c:\\Users\\Admin\\Desktop\\rag-assistant-project\\backend\\data\\vector_store",
  "num_chunks": 105
}
